# Topic 21g - Visualization practice

A consolidation notebook. Everything here uses the whole Topic 21 toolkit and gives you no worked answers - only the checks that tell you whether your answer is right.

Work through the [main lesson](topic21_data_visualization.ipynb) and notebooks 21a to 21f first.

## The scenario

The managing director is opening next week's board meeting with five charts and nothing else - no code, no tables, just the pictures. She has asked you for:

1. Which **segment** is the most profitable, and by how much?
2. Did the **number of orders** grow steadily through 2023, or in bursts?
3. Which **product category** has the widest spread of unit prices - and does that line up with the return-rate problem you already know about from Topic 20?
4. Which **day-and-channel combination** is busiest, and which is quietest?
5. How concentrated is profit among the best customers - does it matter that a third of customers are "Gold" tier if that tier holds nearly all the profit?

Every one of these is a different chart type for a reason - use the chart-selection rule from the main lesson to pick each one.

In [1]:
# Run this cell as it is.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


def find_data_dir():
    """Locate the lesson's data folder from the repository root, the lesson
    folder, or the assignments folder, so this notebook runs from anywhere."""
    candidates = [
        Path.cwd() / "data",
        Path.cwd() / "data_visualization" / "data",
        Path.cwd().parent / "data",
    ]
    for candidate in candidates:
        if (candidate / "orders.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find orders.csv. Expected a data folder containing the six "
        "Topic 20/21 tables. Looked in: " + ", ".join(str(c) for c in candidates)
    )


DATA_DIR = find_data_dir()

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 130)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.color"] = "#e1e0d9"
plt.rcParams["grid.linewidth"] = 0.8
plt.rcParams["axes.edgecolor"] = "#c3c2b7"
plt.rcParams["axes.axisbelow"] = True

# The course palette. Hues are assigned in this fixed order to every chart in
# Topic 21 - never cycled, never re-generated - so a category keeps the same
# colour in every chart it appears in. See 21a for why this matters.
PALETTE = {
    "blue": "#2a78d6", "aqua": "#1baf7a", "yellow": "#eda100", "green": "#008300",
    "violet": "#4a3aa7", "red": "#e34948", "magenta": "#e87ba4", "orange": "#eb6834",
}
MUTED = "#c3c2b7"    # de-emphasis grey, for "everything but the finding"
INK = "#0b0b0b"      # text and value labels

# Fixed colour assignment per category, used throughout this topic.
ZONE_COLOR = {"Coastal": PALETTE["blue"], "Middle": PALETTE["aqua"], "Northern": PALETTE["yellow"]}
CHANNEL_COLOR = {"Field Agent": PALETTE["blue"], "Online": PALETTE["aqua"],
                 "Phone": PALETTE["yellow"], "Shop": PALETTE["green"]}
SEGMENT_COLOR = {"Retail": PALETTE["blue"], "Wholesale": PALETTE["aqua"], "Institution": PALETTE["yellow"]}
CATEGORY_COLOR = {
    "Beverages": PALETTE["blue"], "Grains": PALETTE["aqua"], "Cooking Oils": PALETTE["yellow"],
    "Personal Care": PALETTE["green"], "Household": PALETTE["violet"], "Snacks": PALETTE["red"],
    "Canned Goods": PALETTE["magenta"],
}

print("pandas version:   ", pd.__version__)
print("data folder:      ", DATA_DIR)
print("files:            ", sorted(p.name for p in DATA_DIR.glob("*.csv")))

pandas version:    3.0.6
data folder:       /home/johnny/Projects/itfy-da/data_visualization/data
files:             ['customers.csv', 'monthly_targets_wide.csv', 'orders.csv', 'products.csv', 'regions.csv', 'returns.csv']


In [2]:
# Paste the function you built in Topic 20. This is the same contract: five
# raw tables in, one analysis-ready table out, with its own row-count check.
def clean_region(series):
    return (series.str.strip()
                  .str.replace(r"\s+", " ", regex=True)
                  .str.title())


def build_analysis_table(orders, customers, products, regions, returns):
    """Combine the five raw tables into one analysis-ready table (Topic 20)."""
    expected_rows = len(orders)

    customers = customers.copy()
    customers["region"] = clean_region(customers["region"])
    products = products.drop_duplicates(subset="product_id", keep="first")
    customers = customers.merge(regions, on="region", how="left", validate="m:1")

    table = (orders
             .merge(customers, on="customer_id", how="left", validate="m:1")
             .merge(products, on="product_id", how="left", validate="m:1"))

    returned_ids = set(returns["order_id"])
    table["was_returned"] = table["order_id"].isin(returned_ids)

    table["order_date"] = pd.to_datetime(table["order_date"])
    table["month"] = table["order_date"].dt.strftime("%b")
    table["weekday"] = table["order_date"].dt.day_name()
    table["revenue_ghs"] = (table["quantity"] * table["unit_price_ghs"]).round(2)
    table["cost_ghs"] = (table["quantity"] * table["unit_cost_ghs"]).round(2)
    table["profit_ghs"] = (table["revenue_ghs"] - table["cost_ghs"]).round(2)

    if len(table) != expected_rows:
        raise ValueError(
            f"Row count changed during wrangling: {expected_rows} -> {len(table)}. "
            "A merge has duplicated rows."
        )
    return table.sort_values("order_date").reset_index(drop=True)


orders = pd.read_csv(DATA_DIR / "orders.csv")
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
regions = pd.read_csv(DATA_DIR / "regions.csv")
returns = pd.read_csv(DATA_DIR / "returns.csv")
targets_wide = pd.read_csv(DATA_DIR / "monthly_targets_wide.csv")

analysis = build_analysis_table(orders, customers, products, regions, returns)
print(f"Analysis table: {analysis.shape[0]} rows x {analysis.shape[1]} columns")
print(f"Total revenue: GHS {analysis['revenue_ghs'].sum():,.2f}")
print(f"Total profit:  GHS {analysis['profit_ghs'].sum():,.2f}")

Analysis table: 1200 rows x 24 columns
Total revenue: GHS 230,001.08
Total profit:  GHS 65,886.38


In [3]:
import seaborn as sns

charts_dir = Path("charts")
charts_dir.mkdir(exist_ok=True)
print("Analysis table ready:", analysis.shape)

Analysis table ready: (1200, 24)


---

## Task 1 - Profit by segment

Build a sorted bar chart of total profit by segment, with the top segment emphasised in the course blue and the others in grey. Title it with the finding, label both axes, and add value labels on the bars.

In [4]:
# YOUR CODE HERE

## Task 2 - Orders over time

Build a line chart of the **number of orders per month** in 2023 (not revenue), with a 3-month rolling average overlaid in the course blue and the raw monthly line in grey. Title it with the finding.

Look carefully at the raw line before you title the chart: find the single busiest month using `.idxmax()`, then check whether any other month ties it. A title that says "busiest month was X" when two months are tied is wrong, even if the code that produced it ran without error.

In [5]:
# YOUR CODE HERE

## Task 3 - Distribution of unit price by category

Build a box plot of `unit_price_ghs` by category, sorted by median. Identify the category with the highest median unit price, and in a markdown cell below, connect it to a fact from Topic 20: that category's return rate.

In [6]:
# YOUR CODE HERE

**Is the category with the highest unit price also the category with the highest return rate? Write the connection, with both numbers.**

*(Write your answer here.)*

## Task 4 - Busiest day and channel

Build a sequential heatmap of order **count** by weekday (rows) and channel (columns). Use one hue, light to dark - this is a magnitude grid, not a correlation, so it does not get the diverging treatment from the main lesson's correlation heatmap. Annotate each cell with its count.

In [7]:
# YOUR CODE HERE

## Task 5 - Profit concentration among customers

Using `pd.qcut`, split customers into three profit tiers - `Bronze`, `Silver`, `Gold` - exactly as Topic 20g did. Compute each tier's share of total profit, and build **one** chart of your choice (bar or pie) to show it. Justify your choice in a one-line comment: is this a case where a pie chart is defensible, by the rule from 21f?

In [8]:
# YOUR CODE HERE

---

## Task 6 - The board page

Assemble Tasks 1, 2, 3, 4, and 5 into a single Figure with `plt.subplots(2, 3)` (five panels used, one left blank or holding a title/summary), save it to `charts/board_dashboard.png` at 150 dpi, and display it.

In [9]:
# YOUR CODE HERE

## Self-check

- [ ] Task 1: the most profitable segment is Retail, at GHS 33,442.04
- [ ] Task 2: August and November are **tied** for the busiest month, at 115 orders each - your title does not claim a single busiest month unless it checked for ties
- [ ] Task 3: Cooking Oils has the highest median unit price (GHS 76.33), and it is also the category with the highest return rate (16.5%) from Topic 20
- [ ] Task 4: Shop on Thursday is the single busiest weekday-channel combination, at 90 orders
- [ ] Task 5: Gold-tier customers (the top third by profit) hold 69.2% of all profit, despite being only a third of customers
- [ ] Every chart has a title that states a finding, axis labels with units, and - where there is more than one series - a legend
- [ ] No `# YOUR CODE HERE` remains anywhere
- [ ] The board dashboard is saved to `charts/board_dashboard.png`

---
## Where this fits

- [Back to the main lesson](topic21_data_visualization.ipynb)
- Previous: [21f - Composition and seaborn](topic21f_composition_and_seaborn.ipynb)
- Next: [Topic 21 assignment](assignments/topic21_assignment_trading_dashboard.ipynb)